In [25]:
# train_one_piece.py
import os
import random
from pathlib import Path
from typing import Dict, Any
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
import torchvision.models as models
import pandas as pd
from sklearn.metrics import f1_score
from tqdm import tqdm

In [ ]:

# -----------------------
# Config
# -----------------------
CFG = {
    "train_dir": "data/train",          # Папка с обучающими данными, структура: train/class_name/*.jpg
    "test_dir": "data/test",            # Папка с тестовыми изображениями
    "num_classes": 18,                  # Количество классов (персонажей)
    "seed": 42,                          # Фиксируем seed для воспроизводимости
    "img_size": 224*2,                   # Размер изображений для модели
    "batch_size": 64,                     # Размер batch
    "epochs": 20,                         # Количество эпох
    "lr": 3e-4,                           # learning rate
    "weight_decay": 1e-4,                 # weight decay для оптимизатора
    "device": "cuda" if torch.cuda.is_available() else "cpu",  # Выбор устройства
    "save_dir": "checkpoints",           # Папка для сохранения модели
}

# Создание папки для чекпоинтов
os.makedirs(CFG["save_dir"], exist_ok=True)
torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
random.seed(CFG["seed"])



In [ ]:

# -----------------------
# Dataset
# -----------------------
class AnimeCharactersDataset(Dataset):
    def __init__(self, root_dir: str, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform

        # Собираем имена папок как классы
        self.classes = sorted([d.name for d in self.root_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        print("Classes found:", self.classes)

        # Собираем все изображения с их метками
        self.samples = []
        for cls in self.classes:
            cls_dir = self.root_dir / cls
            label = self.class_to_idx[cls]
            for img_file in cls_dir.glob("*"):
                if img_file.suffix.lower() in [".jpg", ".png", ".jpeg"]:
                    self.samples.append((str(img_file), label))
        print(f"Total samples found: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert("RGB")
        except Exception as e:
            print(f"Error loading {path}: {e}")
        if self.transform:
            img = self.transform(img)
        return img, label


In [ ]:

# -----------------------
# Transforms
# -----------------------
train_transform = T.Compose([
    T.Resize((CFG["img_size"], CFG["img_size"])),    # Изменяем размер
    T.RandomHorizontalFlip(0.5),                     # Случайное отражение
    T.RandomRotation(15),                            # Случайный поворот
    T.ColorJitter(0.2, 0.2, 0.2, 0.05),             # Цветовые вариации
    T.RandomPerspective(0.1, 0.3),                   # Перспективное искажение
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225]),            # Нормализация по ImageNet
])

val_transform = T.Compose([
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225]),
])



In [ ]:

# -----------------------
# Data Loaders
# -----------------------
full_ds = AnimeCharactersDataset(root_dir=CFG["train_dir"], transform=train_transform)

# Делим на train/val 80/20
val_size = int(0.2 * len(full_ds))
train_size = len(full_ds) - val_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size], generator=torch.Generator().manual_seed(CFG["seed"]))
val_ds.dataset.transform = val_transform

# DataLoader с num_workers=0 для простоты отладки
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=0)

print(f"Num training batches: {len(train_loader)}, Num validation batches: {len(val_loader)}")


Classes found: ['Ace', 'Akainu', 'Brook', 'Chopper', 'Crocodile', 'Franky', 'Jinbei', 'Kurohige', 'Law', 'Luffy', 'Mihawk', 'Nami', 'Rayleigh', 'Robin', 'Sanji', 'Shanks', 'Usopp', 'Zoro']
Total samples found: 2915
Num training batches: 37, Num validation batches: 10


In [ ]:
# -----------------------
# Модель (ResNet-18)
# -----------------------
def build_model(num_classes: int):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    n_in = model.fc.in_features
    model.fc = nn.Linear(n_in, num_classes)  # меняем последний слой под количество классов
    return model

# Проверка forward pass
device = CFG["device"]
model = build_model(CFG["num_classes"]).to(device)
print("Device:", device)
dummy = torch.randn(1, 3, CFG["img_size"], CFG["img_size"]).to(device)
try:
    out = model(dummy)
    print("Forward pass OK, output shape:", out.shape)
except Exception as e:
    print("Error in forward pass:", e)

# -----------------------
# Training functions
# -----------------------
def train_one_epoch(model, loader, loss_fn, optimizer, device, epoch):
    model.train()
    losses, preds_all, targets_all = [], [], []

    print(f"\n[Epoch {epoch}] Training...")
    for i, (imgs, targets) in enumerate(tqdm(loader, desc="Training batches", leave=False)):
        imgs, targets = imgs.to(device), targets.to(device)
        logits = model(imgs)
        loss = loss_fn(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        preds_all.extend(logits.argmax(1).cpu().numpy())
        targets_all.extend(targets.cpu().numpy())

    train_loss = np.mean(losses)
    train_f1 = f1_score(targets_all, preds_all, average="macro")
    print(f"Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f}")
    return train_loss, train_f1

def validate(model, loader, loss_fn, device, epoch):
    model.eval()
    losses, preds_all, targets_all = [], [], []

    print(f"[Epoch {epoch}] Validation...")
    with torch.no_grad():
        for i, (imgs, targets) in enumerate(tqdm(loader, desc="Validation batches", leave=False)):
            imgs, targets = imgs.to(device), targets.to(device)
            logits = model(imgs)
            loss = loss_fn(logits, targets)

            losses.append(loss.item())
            preds_all.extend(logits.argmax(1).cpu().numpy())
            targets_all.extend(targets.cpu().numpy())

    val_loss = np.mean(losses)
    val_f1 = f1_score(targets_all, preds_all, average="macro")
    print(f"Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f}")
    return val_loss, val_f1


Device: cuda
Forward pass OK, output shape: torch.Size([1, 18])


In [ ]:

# -----------------------
# Main loop
# -----------------------
def main(cfg: Dict[str, Any]):
    device = cfg["device"]
    model = build_model(cfg["num_classes"]).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"])
    loss_fn = nn.CrossEntropyLoss()

    best_f1 = 0
    print("Starting training...")

    for epoch in range(1, cfg["epochs"] + 1):
        tr_loss, tr_f1 = train_one_epoch(model, train_loader, loss_fn, optimizer, device, epoch)
        val_loss, val_f1 = validate(model, val_loader, loss_fn, device, epoch)
        scheduler.step()

        print(f"Epoch {epoch} summary | Train F1: {tr_f1:.4f} | Val F1: {val_f1:.4f}")

        # Сохраняем лучшую модель
        if val_f1 > best_f1:
            best_f1 = val_f1
            save_path = f"{cfg['save_dir']}/best_resnet18.pt"
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "val_f1": val_f1
            }, save_path)
            print(f"Saved new best model to {save_path}")

    print("\nTraining complete. Best F1 =", best_f1)


In [ ]:
# -----------------------
# Test Dataset
# -----------------------
class TestDataset(Dataset):
    def __init__(self, root_dir: str, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        for img_file in sorted(self.root_dir.glob("*")):
            if img_file.suffix.lower() in [".jpg", ".png", ".jpeg"]:
                self.samples.append(str(img_file))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path = self.samples[idx]
        img_id = Path(path).stem
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, img_id

# -----------------------
# Inference on test set
# -----------------------
def predict_and_save(model, test_dir, submission_path, device, class_to_idx):
    test_ds = TestDataset(test_dir, transform=val_transform)
    test_loader = DataLoader(test_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=0)

    model.eval()
    results = []

    print(f"Predicting {len(test_ds)} test images...")
    with torch.no_grad():
        for imgs, ids in tqdm(test_loader, desc="Predicting"):
            imgs = imgs.to(device)
            logits = model(imgs)
            preds = logits.argmax(1).cpu().numpy()
            for img_id, pred in zip(ids, preds):
                results.append({"id": img_id, "label": int(pred)})

    df = pd.DataFrame(results)
    df.to_csv(submission_path, index=False)
    print(f"Submission saved to {submission_path}")

    


In [ ]:
if __name__ == "__main__":
    main(CFG)
    
    # Загрузим лучшую модель для теста
    best_model_path = f"{CFG['save_dir']}/best_resnet18.pt"
    checkpoint = torch.load(best_model_path, map_location=CFG["device"])
    model = build_model(CFG["num_classes"]).to(CFG["device"])
    model.load_state_dict(checkpoint["model_state"])

    # Предсказания на тесте
    predict_and_save(model, test_dir=CFG["test_dir"], submission_path="submission.csv",
                        device=CFG["device"], class_to_idx=full_ds.class_to_idx)

Starting training...

[Epoch 1] Training...


Train Loss: 1.2659 | Train F1: 0.6663
[Epoch 1] Validation...


Validation batches:   0%|          | 0/10 [00:00<?, ?it/s]c:\Users\Kostya\AppData\Local\Programs\Python\Python312\Lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Val Loss: 0.8226 | Val F1: 0.7872
Epoch 1 summary | Train F1: 0.6663 | Val F1: 0.7872
Saved new best model to checkpoints/best_resnet18.pt

[Epoch 2] Training...


Training batches:   8%|▊         | 3/37 [00:16<03:08,  5.54s/it]c:\Users\Kostya\AppData\Local\Programs\Python\Python312\Lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


...

Train Loss: 0.0018 | Train F1: 1.0000
[Epoch 20] Validation...


Validation batches:   0%|          | 0/10 [00:00<?, ?it/s]c:\Users\Kostya\AppData\Local\Programs\Python\Python312\Lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
C:\Users\Kostya\AppData\Local\Temp\ipykernel_28976\2454529262.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`.

Val Loss: 0.1623 | Val F1: 0.9541
Epoch 20 summary | Train F1: 1.0000 | Val F1: 0.9541

Training complete. Best F1 = 0.956044636354084
Predicting 849 test images...


Predicting:   0%|          | 0/14 [00:00<?, ?it/s]c:\Users\Kostya\AppData\Local\Programs\Python\Python312\Lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Predicting: 100%|██████████| 14/14 [00:09<00:00,  1.47it/s]


PermissionError: [Errno 13] Permission denied: 'submission.csv'